# 12: Trajectory Profiling & Engagement

**Semantic reasoning alternative to ML-based trajectory classification.**

---

## Competency Questions Addressed

| CQ | Question | Method |
|-----|----------|--------|
| #7 | Trajectory classification based on shared characteristics | Existing KG properties (purpose, navigation style) + materialized engagement levels |
| #8 | Trajectory clustering based on patterns | Grouping by shared purpose + movement style |
| #9 | Recognition of visitor engagement to cultural space | Engagement levels (High/Medium/Low) from interaction type counts |

---

## Contents

1. [Setup](#1-setup)
2. [SHACL Validation](#2-shacl-validation)
3. [Existing KG Properties](#3-existing-kg-properties)
4. [POI Relationships (SR-05, SR-06)](#4-poi-relationships)
5. [Engagement Levels (SR-10)](#5-engagement-levels)
6. [Distribution Analysis](#6-distribution-analysis)
7. [Profile Grouping](#7-profile-grouping)
8. [Cross-Tabulation](#8-cross-tabulation)
9. [Spot-Check Verification](#9-spot-check-verification)
10. [Engagement by Purpose (CQ #9)](#10-engagement-by-purpose)

In [ ]:
# 1. Setup
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import seaborn as sns
from helpers.semantic_reasoning import SemanticReasoningEngine

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

engine = SemanticReasoningEngine()
print('SemanticReasoningEngine initialized.')
print(f'  Query endpoint: {engine.query_endpoint}')
print(f'  Update endpoint: {engine.update_endpoint}')

---

## 2. SHACL Validation

Validate data quality before reasoning. Shapes check:
- Trajectories have navigation patterns, purposes, temporal contexts
- Interactions link to exactly one POI with positive dwell time
- Positions belong to exactly one trajectory

In [ ]:
# 2. SHACL Validation
print('Running SHACL validation...')
print('=' * 60)

report = engine.run_validation()

print(f"Conforms: {report['conforms']}")
print(f"Violations: {report['violations']}")
print(f"Warnings: {report['warnings']}")
print(f"Total issues: {report['total']}")

if report['results']:
    # Group by severity and message
    from collections import Counter
    msg_counts = Counter((r['severity'], r['message']) for r in report['results'])

    print('\nIssue summary:')
    for (severity, message), count in msg_counts.most_common(10):
        print(f'  [{severity}] {message}: {count} occurrences')

    if report['total'] > 10:
        print(f'  ... and {report["total"] - 10} more')
else:
    print('\nNo issues found. Data quality baseline passed.')

---

## 3. Existing KG Properties

The pipeline (notebooks 1-9) already materializes purpose, navigation style, and temporal
context for each trajectory. No profiling rules needed — we query these directly.

- `trajecto:hasPurpose` — assigned by notebook 7
- `patterns:exhibitsPattern` — assigned by notebook 6
- `context:hasTemporalContext` — assigned by notebook 3

In [ ]:
# 3. Show existing KG properties (no materialization needed)
print('Clearing previous materialized triples...')
engine.clear_reasoning_triples()

all_trajs = engine.get_all_trajectories()
print(f'\nTotal trajectories: {len(all_trajs)}')

# Purpose distribution (already in KG)
purpose_dist = engine.get_purpose_distribution()
print(f'\nPurpose distribution ({len(purpose_dist)} purposes):')
for purpose, count in list(purpose_dist.items())[:15]:
    bar = '#' * min(int(count / 5), 40)
    print(f'  {purpose:<45} {count:>4} {bar}')

# Sample trajectory profile
for t in all_trajs[:5]:
    profile = engine.get_trajectory_profile(t)
    if profile.get('purpose'):
        traj_name = t.split('#')[-1][:40]
        print(f'\nSample profile: {traj_name}')
        for k, v in profile.items():
            if isinstance(v, list):
                v_str = ', '.join(str(x).split(':')[-1] if ':' in str(x) else str(x) for x in v)
            else:
                v_str = str(v).split(':')[-1] if ':' in str(v) else str(v)
            print(f'  {k}: {v_str}')
        break

---

## 4. POI Relationships (SR-05, SR-06)

- **SR-05**: Co-visitation — dominant POI linked to other meaningful POIs in same trajectory (`poi:covisitedWith`)
- **SR-06**: Sequence succession — POI A followed by POI B in position chain (`poi:followedBy`)

In [ ]:
# 4. POI Relationships
print('Executing relationship rules (SR-05, SR-06)...')
print('=' * 60)
rel_results = engine.execute_relationship_rules()

for rule, success in rel_results.items():
    status = 'OK' if success else 'FAILED'
    print(f'  {rule}: {status}')

stats = engine.get_relationship_stats()
print('\nMaterialized relationships:')
print(f'  Co-visitation links (poi:covisitedWith): {stats["covisitation"]}')
print(f'  Sequence links (poi:followedBy): {stats["sequence"]}')

# Show sample co-visitation
covisit_df = engine.get_covisitation_links(limit=10)
if not covisit_df.empty:
    print('\nSample co-visitation links:')
    for _, row in covisit_df.head(5).iterrows():
        poi_a = row['poi'].split('#')[-1][:25] if '#' in row['poi'] else row['poi'][-25:]
        poi_b = row['covisited'].split('#')[-1][:25] if '#' in row['covisited'] else row['covisited'][-25:]
        print(f'  {poi_a} <-> {poi_b}')

# Show sample sequence links
seq_df = engine.get_sequence_links(limit=10)
if not seq_df.empty:
    print('\nSample sequence links:')
    for _, row in seq_df.head(5).iterrows():
        poi_a = row['poi'].split('#')[-1][:25] if '#' in row['poi'] else row['poi'][-25:]
        poi_b = row['next'].split('#')[-1][:25] if '#' in row['next'] else row['next'][-25:]
        print(f'  {poi_a} -> {poi_b}')

---

## 5. Engagement Levels (SR-10) — CQ #9

- **High**: >= 3 immersive interactions
- **Medium**: >= 5 meaningful, < 3 immersive
- **Low**: remainder with any interactions

Materializes `trajecto:hasEngagementLevel` with values `trajecto:HighEngagement`, `trajecto:MediumEngagement`, `trajecto:LowEngagement`.

In [ ]:
# 5. Engagement Levels
print('Executing engagement rules (SR-10)...')
print('=' * 60)
eng_results = engine.execute_engagement_rules()

for rule, success in eng_results.items():
    status = 'OK' if success else 'FAILED'
    print(f'  {rule}: {status}')

eng_dist = engine.get_engagement_distribution()
print('\nEngagement level distribution:')
total_eng = sum(eng_dist.values())
for level, count in sorted(eng_dist.items()):
    pct = (count / total_eng * 100) if total_eng > 0 else 0
    bar = '#' * int(pct / 2)
    print(f'  {level:<35} {count:>4} ({pct:5.1f}%) {bar}')

---

## 6. Distribution Analysis

Charts for purpose distribution and engagement levels.

In [ ]:
# 6. Distribution charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Purpose distribution
if purpose_dist:
    ax = axes[0]
    labels = [p.split(':')[-1] if ':' in p else p.split('#')[-1] for p in purpose_dist]
    counts = list(purpose_dist.values())
    y_pos = range(len(labels))
    ax.barh(y_pos, counts, color='#3498db', edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Trajectory Count')
    ax.set_title('Purpose Distribution', fontweight='bold')
    ax.invert_yaxis()

# Engagement levels
if eng_dist:
    ax = axes[1]
    eng_labels = [l.split(':')[-1] if ':' in l else l for l in eng_dist]
    eng_counts = list(eng_dist.values())
    colors = {'HighEngagement': '#e74c3c', 'MediumEngagement': '#f39c12', 'LowEngagement': '#95a5a6'}
    bar_colors = [colors.get(l, '#3498db') for l in eng_labels]
    ax.bar(eng_labels, eng_counts, color=bar_colors, edgecolor='white')
    ax.set_ylabel('Trajectory Count')
    ax.set_title('Engagement Levels', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Trajectories with purpose: {sum(purpose_dist.values())}')
print(f'Distinct purposes: {len(purpose_dist)}')

---

## 7. Profile Grouping — CQ #8

Group trajectories by shared semantic profile: purpose + navigation style.
This is the semantic alternative to numerical clustering.

In [ ]:
# 7. Profile grouping
print('Semantic profile groups (purpose + navigation style):')
print('=' * 60)

groups = engine.get_profile_groups()

if groups:
    print(f'Total distinct profiles: {len(groups)}')
    print('\nTop 15 profile groups:')
    print(f'{"Purpose":<40} {"Navigation":<25} {"Count":>5}')
    print('-' * 75)
    for g in groups[:15]:
        pur = g['purpose'].split(':')[-1] if ':' in g['purpose'] else g['purpose']
        nav = g['navigationStyle'].split(':')[-1] if ':' in g['navigationStyle'] else g['navigationStyle']
        print(f'{pur:<40} {nav:<25} {g["count"]:>5}')

    # Size distribution of groups
    sizes = [g['count'] for g in groups]
    print('\nGroup size statistics:')
    print(f'  Largest: {max(sizes)}')
    print(f'  Smallest: {min(sizes)}')
    print(f'  Median: {sorted(sizes)[len(sizes)//2]}')
    print(f'  Singletons: {sum(1 for s in sizes if s == 1)}')
else:
    print('No profile groups found.')

---

## 8. Cross-Tabulation

Purpose vs navigation style.

In [ ]:
# 8. Cross-tabulation heatmap
cross_tab = engine.get_cross_tabulation()

if not cross_tab.empty:
    # Clean column and index names
    cross_tab.columns = [c.split(':')[-1] if ':' in c else c for c in cross_tab.columns]
    cross_tab.index = [i.split(':')[-1] if ':' in i else i for i in cross_tab.index]

    fig, ax = plt.subplots(figsize=(12, max(6, len(cross_tab) * 0.4)))
    sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
                linewidths=0.5, cbar_kws={'label': 'Trajectory Count'})
    ax.set_title('Purpose vs Navigation Style', fontsize=14, fontweight='bold')
    ax.set_xlabel('Navigation Style')
    ax.set_ylabel('Purpose')
    plt.tight_layout()
    plt.show()
else:
    print('No cross-tabulation data available.')

---

## 9. Spot-Check Verification

Verify that 3 trajectories' profiles match their actual POI interactions.

In [ ]:
# 9. Spot-check verification
print('Spot-check: verifying profiles against actual interactions')
print('=' * 60)

checked = 0
for traj_uri in all_trajs:
    if checked >= 3:
        break

    check = engine.spot_check_profile(traj_uri)
    profile = check['profile']
    interactions = check['interactions']

    if not profile or interactions.empty:
        continue

    checked += 1
    traj_name = traj_uri.split('#')[-1][:40]
    print(f'\n--- Trajectory: {traj_name} ---')

    # Show profile
    print('  Profile:')
    for k, v in profile.items():
        if isinstance(v, list):
            v_str = ', '.join(str(x).split(':')[-1] if ':' in str(x) else str(x) for x in v)
        else:
            v_str = str(v).split(':')[-1] if ':' in str(v) else str(v)
        print(f'    {k}: {v_str}')

    # Show actual interactions
    print(f'  Actual interactions ({len(interactions)} total):')
    for _, row in interactions.head(8).iterrows():
        cat = row['cat'].split('#')[-1] if '#' in row['cat'] else row['cat']
        int_type = row['intType'].split('#')[-1] if '#' in row['intType'] else row['intType']
        dwell = f"{float(row.get('dwell', 0)):.0f}s" if row.get('dwell') else 'N/A'
        print(f'    [{int_type:<25}] {cat:<20} dwell={dwell}')
    if len(interactions) > 8:
        print(f'    ... and {len(interactions) - 8} more')

if checked == 0:
    print('No trajectories with profiles found for spot-checking.')

---

## 10. Engagement by Purpose — CQ #9

How does engagement level vary by trajectory purpose?
Cultural trajectories with more immersive interactions should show higher engagement.

In [ ]:
# 10. Engagement by purpose
eng_by_purpose = engine.get_engagement_by_purpose()

if not eng_by_purpose.empty:
    # Clean names
    eng_by_purpose.columns = [c.split(':')[-1] if ':' in c else c for c in eng_by_purpose.columns]
    eng_by_purpose.index = [i.split(':')[-1] if ':' in i else i for i in eng_by_purpose.index]

    # Sort by total count
    eng_by_purpose['Total'] = eng_by_purpose.sum(axis=1)
    eng_by_purpose = eng_by_purpose.sort_values('Total', ascending=False)
    plot_data = eng_by_purpose.drop(columns='Total')

    fig, ax = plt.subplots(figsize=(14, max(6, len(plot_data) * 0.4)))

    color_map = {
        'HighEngagement': '#e74c3c',
        'MediumEngagement': '#f39c12',
        'LowEngagement': '#95a5a6'
    }
    ordered_cols = [c for c in ['HighEngagement', 'MediumEngagement', 'LowEngagement'] if c in plot_data.columns]
    colors = [color_map.get(c, '#3498db') for c in ordered_cols]

    plot_data[ordered_cols].plot(kind='barh', stacked=True, ax=ax, color=colors, edgecolor='white')
    ax.set_xlabel('Trajectory Count')
    ax.set_title('Engagement Level by Purpose (CQ #9)', fontsize=14, fontweight='bold')
    ax.legend(title='Engagement', loc='lower right')
    plt.tight_layout()
    plt.show()

    # Print table
    print('\nEngagement breakdown by purpose:')
    print(eng_by_purpose.to_string())
else:
    print('No engagement-by-purpose data available.')

In [ ]:
# Summary
print('\n' + '=' * 60)
print('TRAJECTORY PROFILING COMPLETE')
print('=' * 60)

all_counts = engine.count_materialized_triples()
total = sum(all_counts.values())
print(f'\nMaterialized triples: {total}')
for pred, count in all_counts.items():
    print(f'  {pred}: {count}')

print(f'\nProfile groups: {len(groups) if groups else 0}')
print(f'Engagement distribution: {eng_dist}')
print('\nNext: 13-owl-classification.ipynb')